# MOFF2 Training Workflow

This folder contains the scripts used to train and locally refine the MOFF2
coarse-grained force field. The workflow has five stages:

1. Build mixed noise/data ensembles.
2. Compute reduced-energy basis terms for the MOFF2 energy function.
3. Train the global model by contrastive learning.
4. Prepare FEP/reweighting inputs from production or validation simulations.
5. Apply ESS-constrained FEP refinement against target observables.

The training scripts import MOFF2 through the OpenABC namespace:

```python
from openabc.forcefields.MOFF2.core import compute_PE
from openabc.forcefields.MOFF2.forcefields import HPSMOFFCosAngleTestModel
```

Before running scripts directly from this repository, make sure the repository
root is on `PYTHONPATH`, or install OpenABC in editable mode.

```bash
export PYTHONPATH=/path/to/openabc:$PYTHONPATH
```

## Requirements

The workflow assumes a Python environment with OpenABC and the scientific
simulation stack used by the scripts:

- `openabc`
- `openmm`
- `mdtraj`
- `numpy`
- `pandas`
- `torch`
- `FastMBAR`
- `scipy`
- `matplotlib`
- `seaborn`
- `tqdm`

GPU support is used in the Potential Contrasting training `step3_train_pc.py` and FEP/ESS fine-tuning stages `step5_ess_ah_gau_density_all_idp_mdp_op.py`.  currently
expects two GPUs on one node.

Required input files include:

reference CA trajectories;
noise simulations;
training-input/;
parameters/; # CSV files such as raw_MJ.csv and HPS_Urry_parameters.csv;
simulation outputs used for FEP refinement.

MOFF2 was trained on files below: 

- reference CA trajectories;
- noise simulations;
- `parameters/`; parameters used as starting point for MOFF2.
- simulation outputs used for FEP/ESS refinement.


## Stage 1: Build Mixed Noise/Data Ensembles

Script:

```text
step1_compute_noise_u0.py
```

Purpose:

This step combines noise samples and reference data samples for one protein,
computes their reduced energies under the mixed noise ensemble, and writes the
basic training input files.

Main inputs:

- reference CA trajectory and CA PDB for the selected protein;
- noise simulation trajectories from umbrella-biased HPS or HPS-SBM runs;
- the corresponding unbiased OpenMM system XML.

Example:

```bash
python step1_compute_noise_u0.py \
  --protein ACTR \
  --n0 50000 \
  --n1 50000 \
  --T0 300.0
```

Important arguments:

- `--protein`: protein name.
- `--n0`: target number of noise samples.
- `--n1`: target number of data/reference samples.
- `--T0`: noise simulation temperature in K.

Outputs:

```text
training-input/n0-<n0>-n1-<n1>/<protein>/
  <protein>_ca.pdb
  traj.dcd
  labels.npy
  u0.npy
```

`labels.npy` marks noise samples as `0` and data samples as `1`. `u0.npy`
contains the reduced energy of each sample under the mixed reference/noise
ensemble.

## Stage 2: Compute MOFF2 Basis Terms

Scripts:

```text
step2_multi_group_compute_basis_ah_density_spl.py
step2_multi_group_compute_basis_ext_OPs_ah_density_spl.py
step2_multi_group_compute_basis_MDP_ah_density_spl.py
```

Purpose:

This step converts the trajectories from Stage 1 into reduced-energy basis
terms for the MOFF2 energy function. The basis terms include:

- bonded baseline terms;
- Debye-Huckel electrostatics;
- AH pairwise contact basis;
- Gaussian first-solvation-shell basis;
- density-dependent B-spline basis.

For IDPs, OPs, and Evo proteins, use:

```bash
python step2_multi_group_compute_basis_ah_density_spl.py \
  --protein ACTR \
  --n0 50000 \
  --n1 50000 \
  --T1 300.0 \
  --ionic_strength 150 \
  --res_group_mapping default
```

For extended OPs, use:

```bash
python step2_multi_group_compute_basis_ext_OPs_ah_density_spl.py \
  --protein 1soy_clean \
  --n0 50000 \
  --n1 50000 \
  --T1 300.0 \
  --ionic_strength 150 \
  --res_group_mapping default
```

For MDPs, use:

```bash
python step2_multi_group_compute_basis_MDP_ah_density_spl.py \
  --protein Ub2 \
  --n0 10000 \
  --n1 10000 \
  --res_group_mapping default
```

Important arguments:

- `--protein`: protein name.
- `--n0`, `--n1`: sample counts matching Stage 1.
- `--T1`: reference/data temperature in K, for IDP/OP scripts.
- `--ionic_strength`: ionic strength in mM, for IDP/OP scripts.
- `--eta`, `--r0`: density switching parameters.
- `--rho_min`, `--rho_max`: density range for spline basis.
- `--n_internal_knots`: number of internal B-spline knots.
- `--res_group_mapping`: residue grouping used for density basis.

Outputs:

Each script writes a pickle file in the corresponding protein folder:

```text
training-input/n0-<n0>-n1-<n1>/<protein>/
  training_input_ah_density_spl_group_<group>_eta_<eta>_r0_<r0>_rho_range_<rho_min>_<rho_max>_n_internal_knots_<n>_gau.pkl
```

The pickle contains the arrays used by CL training, including:

- `labels`
- `u0`
- `u1_all_bonded`
- `u1_lj_excl`
- `u1_elec`
- `u1_ah_basis`
- `u1_gauss_basis`
- `u1_density_spl_basis`
- density spline metadata
- Gaussian parameters

## Stage 3: Potential-Contrastive Training

Script:

```text
step3_train_pc.py
```

Purpose:

This step trains the global MOFF2 energy model by potential contrasting training across
IDPs, OPs, and MDPs. The optimized parameters include:

- 210 AH pair coefficients;
- 210 Gaussian amplitudes;
- 240 residue/group-dependent density spline coefficients.

The model initializes the AH coefficients from a scaled Miyazawa-Jernigan
matrix and initializes Gaussian coefficients to zero.

Example:

```bash
python -u step3_train_pc.py \
  --n_epochs 10000 \
  --lr 0.5 \
  --MJ_min 0.0 \
  --MJ_max 0.8 \
  --zeta1 0.2 \
  --zeta2 0.004 \
  --res_group_mapping default \
  --IDP_weight 1.0 \
  --OP_weight 1.0 \
  --MDP_weight 0.4 \
  --gauss_delta_mu 0.25 \
  --gauss_width 0.1
```

Important arguments:

- `--MJ_min`, `--MJ_max`: range used to initialize AH coefficients.
- `--zeta1`: L2 regularization for AH and Gaussian coefficient updates.
- `--zeta2`: L2 regularization for density spline coefficients.
- `--IDP_weight`, `--OP_weight`, `--MDP_weight`: class weights in the CL loss.
- `--gauss_delta_mu`, `--gauss_width`: Gaussian shape metadata saved with the model.
- `--backend`: PyTorch distributed backend. Default is `nccl`.

Outputs:

```text
results/group_<group>_IDP_w_<...>_OP_w_<...>_MDP_w_<...>_<knots>_zeta_<zeta1>_<zeta2>_gauss_delta_mu_<...>_gauss_width_<...>/
  results.pkl
  loss.csv
  hydrophobic_scale.pdf
  density_spline.pdf
  pairwise_potentials.pdf
  tests/
```

`results.pkl` is the main potential-contrastive trained parameter file. It contains:

- `hydrophobic_scale`
- `gauss_coeffs`
- `gauss_height_map`
- `spl_coeffs`
- `spl_values`
- density and Gaussian hyperparameters

The script also performs a reweighting-based validation at the end and writes
the results into the `tests/` subdirectory.

## Stage 4: Prepare FEP/ESS fine-tuning Inputs

Script:

```text
step4_prepare_fep_AH_gau_torch.py
```

Purpose:

This step converts simulation trajectories generated with a pc-trained model
into compact FEP input pickles. These files are used by Stage 5 to refine the
parameters without rerunning simulations at every optimization step.

Each protein simulation directory should contain:

```text
<protein_dir>/
  input_parameters.json
  system.xml
  output.dcd
  <protein>_ca.pdb
```

`input_parameters.json` must include at least:

```json
{
  "protein": "A1-LCD+12E",
  "temperature": 298.0,
  "ionic_strength": 150.0,
  "results_pkl": "/path/to/pc/results.pkl"
}
```

Example:

```bash
python step4_prepare_fep_AH_gau_torch.py \
  --protein_dir /path/to/simulation/A1-LCD+12E \
  --output_pkl /path/to/fep_AH_gau/A1-LCD+12E/fep_AH_gau_input.pkl \
  --gauss_delta_mu_nm 0.25 \
  --gauss_width_nm 0.1
```

If `--output_pkl` is omitted, the script writes:

```text
<protein_dir>/fep_AH_input.pkl
```

Stage 5 searches for `fep_AH_gau_input.pkl` first and falls back to
`fep_AH_input.pkl`.

Output keys include:

- `u1_intercept`
- `u1_ah_basis`
- `u1_gauss_basis`
- `u1_density_spl_basis`
- `rg`
- `results_pkl_used`

## Stage 5: FEP/ESS-Constrained FEP Refinement

Script:

```text
step5_ess_ah_gau_density_all_idp_mdp_op.py
```

Purpose:

This step locally refines the pc-trained model using ensemble-averaged target
observables, currently radius of gyration. The optimization uses FEP weights
from the CL reference ensemble and an effective-sample-size penalty to prevent
updates that rely on too few configurations.

Example:

```bash
python step5_ess_ah_gau_density_all_idp_mdp_op.py \
  --results_pkl /path/to/pc/results.pkl \
  --fep_input_root /path/to/fep_AH_gau_inputs \
  --exp_csv exp_plus_sim_a1_ref.csv \
  --output_pkl ess_optimize/results.pkl \
  --output_dir ess_optimize/ \
  --alpha 100.0 \
  --ess0 750.0 \
  --lam_density 1e-2 \
  --lr 1e-3 \
  --n_epochs 500 \
  --device cuda
```

Expected FEP input layout:

```text
<fep_input_root>/
  <protein>/
    fep_AH_gau_input.pkl
```

Expected experimental/target CSV:

The script expects an input CSV containing protein names and target Rg values.
The loaded proteins are matched against the built-in A1-LCD, MDP, and OP lists
in `step5_ess_ah_gau_density_all_idp_mdp_op.py`.

Outputs:

```text
<output_dir>/
  training_log.csv
  loss_curve.pdf
  final_predictions.csv
  ...
<output_pkl>
```

The exported `results.pkl` preserves the original CL parameter structure and
adds the FEP-refined parameter values:

- updated `hydrophobic_scale`
- updated `gauss_coeffs`
- updated `gauss_height_map`
- updated `spl_coeffs`
- updated `spl_values`
- FEP metadata

The script also reruns the reweighting validation using the refined parameter
file and writes results under:

```text
<dirname(output_pkl)>/tests/
```

## Slurm Wrappers

This folder includes Slurm wrappers for the original cluster workflow:

- `run_step1_all.sh`
- `run_step2_multi_group_ah_density_spl_all.sh`
- `run_step2_multi_group_ah_density_spl_*.slurm`
- `run_step3_train_CL.slurm`
- `step4_prepare_fep_AH_gau_torch.slurm`
- `step5_ess_ah_gau_normal_long_plus_sim_ref.slurm`

Treat these as templates. Before submission, check that:

- the conda environment name is correct;
- paths to data and simulation folders are updated;
- the script names match the current canonical names.


## End-to-End Summary

A typical full training/refinement run is:

```bash
# 1. Build mixed noise/data ensemble
python step1_compute_noise_u0.py --protein ACTR --n0 50000 --n1 50000 --T0 300.0

# 2. Compute MOFF2 basis terms
python step2_multi_group_compute_basis_ah_density_spl.py \
  --protein ACTR --n0 50000 --n1 50000 \
  --T1 300.0 --ionic_strength 150 \
  --res_group_mapping default

# 3. Train global CL model
python -u step3_train_CL.py \
  --n_epochs 10000 --lr 0.5 \
  --MJ_min 0.0 --MJ_max 0.8 \
  --zeta1 0.2 --zeta2 0.004 \
  --res_group_mapping default \
  --IDP_weight 1.0 --OP_weight 1.0 --MDP_weight 0.4

# 4. Prepare FEP input from simulations generated by the CL model
python step4_prepare_fep_AH_gau_torch.py \
  --protein_dir /path/to/simulation/A1-LCD+12E \
  --output_pkl /path/to/fep_inputs/A1-LCD+12E/fep_AH_gau_input.pkl

# 5. Run ESS-constrained FEP refinement
python step5_ess_ah_gau_density_all_idp_mdp_op.py \
  --results_pkl /path/to/CL/results.pkl \
  --fep_input_root /path/to/fep_inputs \
  --exp_csv exp_plus_sim_a1_ref.csv \
  --output_pkl ess_results/results.pkl \
  --output_dir ess_results \
  --alpha 100.0 --ess0 750.0 \
  --lam_density 1e-2 \
  --device cuda
```

## Additional Notes 

Trajectories and intermediate training inputs are available upon request. 

# MOFF2 single chain simulations


All simulation related scripts and outputs are available at:

`tutorials/MOFF2-simulations`


## IDP simulations

In [7]:
# %load /home/yumzhang/orcd/pool/work/4-idpcg/OpenABC_moff2_test/tutorials/MOFF2-single-protein/run_test_IDP.py
from pathlib import Path
import json
import shutil

import mdtraj

try:
    import openmm as mm
    import openmm.app as app
    import openmm.unit as unit
except ImportError:
    import simtk.openmm as mm
    import simtk.openmm.app as app
    import simtk.unit as unit

from openabc.forcefields.parsers import HPSParser
from openabc.forcefields.MOFF2.forcefields import MOFF2Model


# =========================
# Clean user inputs
# =========================

protein = "A1-LCD+NLS"

input_pdb = Path(
    "/home/yumzhang/orcd/pool/work/4-idpcg/OpenABC_moff2_test/tutorials/MOFF2-single-protein/"
    "A1-LCD+NLS_ca.pdb"
)

output_dir = Path("outputs") / protein

temperature_K = 298.0
ionic_strength_mM = 150.0
timestep_fs = 10.0

# tiny smoke-test values
n_relax_steps = 50
n_production_steps = 500
report_interval = 10

res_group_mapping_name = "default"
platform_name = "CPU"


# =========================
# Prepare files
# =========================

if not input_pdb.exists():
    raise FileNotFoundError(f"Input PDB not found: {input_pdb}")

output_dir.mkdir(parents=True, exist_ok=True)

ca_pdb = output_dir / f"{protein}_ca.pdb"
shutil.copyfile(input_pdb, ca_pdb)

input_parameters = {
    "protein": protein,
    "input_pdb": str(input_pdb),
    "temperature": temperature_K,
    "ionic_strength": ionic_strength_mM,
    "timestep_fs": timestep_fs,
    "n_relax_steps": n_relax_steps,
    "n_production_steps": n_production_steps,
    "report_interval": report_interval,
    "platform_name": platform_name,
    "res_group_mapping": res_group_mapping_name,
}

with open(output_dir / "input_parameters.json", "w") as f:
    json.dump(input_parameters, f, indent=4)
    f.write("\n")


# =========================
# Build MOFF2 system
# =========================

model = MOFF2Model()
model.append_mol(HPSParser(str(ca_pdb)))

model.protein_bonds.loc[:, "k_bond"] = 8000.0
model.protein_bonds.loc[:, "r0"] = 0.386

ca_traj = mdtraj.load_pdb(str(ca_pdb))
assert ca_traj.n_chains == 1

for _, row in model.atoms.iterrows():
    if row["resname"] == "HIS":
        assert row["charge"] == 0.5

charge = model.atoms["charge"].to_numpy()
charge[0] += 1
charge[-1] -= 1
model.atoms["charge"] = charge

top = app.PDBFile(str(ca_pdb)).getTopology()
model.create_system(top=top, box_a=1000, box_b=1000, box_c=1000)

model.add_protein_bonds(force_group=1)

# Uses packaged:
# openabc/forcefields/MOFF2/forcefields/parameters/MOFF2.pkl
model.add_moff2_forces(
    temperature=temperature_K,
    ionic_strength=ionic_strength_mM,
    res_group_mapping=res_group_mapping_name,
    contact_force_group=2,
    elec_force_group=3,
    density_force_group_start=4,
)

with open(output_dir / "system.xml", "w") as f:
    f.write(mm.XmlSerializer.serialize(model.system))


# =========================
# Run simulation
# =========================

T = temperature_K * unit.kelvin
friction_coeff = 1.0 / unit.picosecond
timestep = timestep_fs * unit.femtosecond

integrator = mm.LangevinMiddleIntegrator(T, friction_coeff, timestep)
init_coord = app.PDBFile(str(ca_pdb)).getPositions()

properties = {"Precision": "mixed"} if platform_name == "CUDA" else {}

model.set_simulation(
    integrator,
    platform_name=platform_name,
    init_coord=init_coord,
    properties=properties,
)

model.simulation.minimizeEnergy()
model.simulation.step(n_relax_steps)

output_dcd = output_dir / "output.dcd"
model.add_reporters(
    report_interval=report_interval,
    output_dcd=str(output_dcd),
)

model.simulation.context.setVelocitiesToTemperature(T)
model.simulation.step(n_production_steps)


# =========================
# Save final outputs
# =========================

checkpoint_path = output_dir / "checkpoint.chk"
model.simulation.saveCheckpoint(str(checkpoint_path))

state = model.simulation.context.getState(
    getPositions=True,
    getVelocities=True,
    enforcePeriodicBox=True,
)

with open(output_dir / "state.xml", "w") as f:
    f.write(mm.XmlSerializer.serialize(state))

final_pdb = output_dir / f"{protein}_final.pdb"
with open(final_pdb, "w") as f:
    app.PDBFile.writeFile(
        model.simulation.topology,
        state.getPositions(),
        f,
    )

print(f"Saved input parameters: {output_dir / 'input_parameters.json'}")
print(f"Saved system XML: {output_dir / 'system.xml'}")
print(f"Saved trajectory: {output_dcd}")
print(f"Saved checkpoint: {checkpoint_path}")
print(f"Saved state XML: {output_dir / 'state.xml'}")
print(f"Saved final PDB: {final_pdb}")


Parse molecule with default settings.
Add protein bonds.
Use platform: CPU
#"Step","Time (ps)","Potential Energy (kJ/mole)","Kinetic Energy (kJ/mole)","Total Energy (kJ/mole)","Temperature (K)","Speed (ns/day)"
60,0.6000000000000003,307.1596497578419,458.9178413044802,766.0774910623221,270.5643605137934,0
70,0.7000000000000004,333.77760458297666,467.69684357163743,801.474448154614,275.74020010985436,27.6
80,0.8000000000000005,381.7993621978196,437.1183802176607,818.9177424154802,257.71204422176373,28.9
90,0.9000000000000006,396.51627662871863,404.8572320907113,801.37350871943,238.69182725308517,29.3
100,1.0000000000000007,383.29333355673043,422.88960231813945,806.1829358748698,249.32317840139456,29.3
110,1.1000000000000008,378.1035474152869,435.26834858214926,813.3718959974362,256.62132038986914,29.3
120,1.2000000000000008,380.983345608267,459.6250859353453,840.6084315436123,270.981330990977,30.1
130,1.300000000000001,411.7079577028239,437.128703130057,848.8366608328809,257.71813030502

## MDP simulations

In [5]:
# %load /home/yumzhang/orcd/pool/work/4-idpcg/OpenABC_moff2_test/tutorials/MOFF2-single-protein/run_test_MDP.py
from pathlib import Path
import json

try:
    import openmm as mm
    import openmm.app as app
    import openmm.unit as unit
except ImportError:
    import simtk.openmm as mm
    import simtk.openmm.app as app
    import simtk.unit as unit

from openabc.forcefields.MOFF2.forcefields import MOFF2Model


# =========================
# Clean user inputs
# =========================

protein = "D14"

reference_dir = Path(
    "/home/yumzhang/orcd/pool/work/4-idpcg/OpenABC_moff2_test/tutorials/MOFF2-single-protein/"
)


aa_pdb = reference_dir / f"{protein}.pdb"
mdp_od_csv = reference_dir / "MDP_OD_info_0_based.csv"

output_dir = Path("outputs") / protein
ca_pdb = output_dir / f"{protein}_ca.pdb"

temperature_K = 283.15
ionic_strength_mM = 156.0
target_rg_nm = None

timestep_fs = 10.0

# Tiny smoke-test values. Increase for production.
n_relax_steps = 50
n_production_steps = 200
report_interval = 50

res_group_mapping_name = "default"
platform_name = "CPU"  # use "CUDA" for GPU production runs


# =========================
# Validate inputs
# =========================

for path in [aa_pdb, mdp_od_csv]:
    if not path.exists():
        raise FileNotFoundError(f"Required input file not found: {path}")

print(f"Protein: {protein}")
print(f"Atomistic PDB: {aa_pdb}")
print(f"MDP OD CSV: {mdp_od_csv}")
print(f"Temperature: {temperature_K} K")
print(f"Ionic strength: {ionic_strength_mM} mM")
print(f"Reference Rg: {target_rg_nm} nm")


# =========================
# Prepare output folder
# =========================

output_dir.mkdir(parents=True, exist_ok=True)

input_parameters = {
    "protein": protein,
    "aa_pdb": str(aa_pdb),
    "ca_pdb": str(ca_pdb),
    "mdp_od_csv": str(mdp_od_csv),
    "temperature": temperature_K,
    "ionic_strength": ionic_strength_mM,
    "target_rg_nm": target_rg_nm,
    "timestep_fs": timestep_fs,
    "n_relax_steps": n_relax_steps,
    "n_production_steps": n_production_steps,
    "report_interval": report_interval,
    "platform_name": platform_name,
    "res_group_mapping": res_group_mapping_name,
}

with open(output_dir / "input_parameters.json", "w") as f:
    json.dump(input_parameters, f, indent=4)
    f.write("\n")


# =========================
# Build MOFF2 MDP system
# =========================

# This constructor:
#   1. parses the atomistic PDB into a CA model;
#   2. reads ordered-domain ranges from MDP_OD_info_0_based.csv;
#   3. keeps angles/dihedrals only within ordered domains;
#   4. keeps native pairs only within the same ordered domain;
#   5. adds bonds, angles, dihedrals, and native pairs.
model = MOFF2Model.from_mdp_pdb(
    aa_pdb=str(aa_pdb),
    ca_pdb=str(ca_pdb),
    mdp_od_csv=str(mdp_od_csv),
    protein_name=protein,
    bond_force_group=1,
    angle_force_group=2,
    dihedral_force_group=3,
    native_pair_force_group=4,
)

#   openabc/forcefields/MOFF2/forcefields/parameters/MOFF2.pkl
# and adds:
#   AH + Gaussian contacts, Debye-Huckel electrostatics, density terms.
model.add_moff2_forces(
    temperature=temperature_K,
    ionic_strength=ionic_strength_mM,
    res_group_mapping=res_group_mapping_name,
    contact_force_group=5,
    elec_force_group=6,
    density_force_group_start=7,
)

with open(output_dir / "system.xml", "w") as f:
    f.write(mm.XmlSerializer.serialize(model.system))


# =========================
# Run short simulation
# =========================

T = temperature_K * unit.kelvin
friction_coeff = 1.0 / unit.picosecond
timestep = timestep_fs * unit.femtosecond

integrator = mm.LangevinMiddleIntegrator(T, friction_coeff, timestep)
init_coord = app.PDBFile(str(ca_pdb)).getPositions()

properties = {"Precision": "mixed"} if platform_name == "CUDA" else {}

model.set_simulation(
    integrator,
    platform_name=platform_name,
    init_coord=init_coord,
    properties=properties,
)

model.simulation.minimizeEnergy()
model.simulation.step(n_relax_steps)

output_dcd = output_dir / "output.dcd"
model.add_reporters(report_interval=report_interval, output_dcd=str(output_dcd))

model.simulation.context.setVelocitiesToTemperature(T)
model.simulation.step(n_production_steps)


# =========================
# Save final outputs
# =========================

checkpoint_path = output_dir / "checkpoint.chk"
model.simulation.saveCheckpoint(str(checkpoint_path))

state = model.simulation.context.getState(
    getPositions=True,
    getVelocities=True,
    enforcePeriodicBox=True,
)

with open(output_dir / "state.xml", "w") as f:
    f.write(mm.XmlSerializer.serialize(state))

final_pdb = output_dir / f"{protein}_final.pdb"
with open(final_pdb, "w") as f:
    app.PDBFile.writeFile(
        model.simulation.topology,
        state.getPositions(),
        f,
    )

print(f"Saved input parameters: {output_dir / 'input_parameters.json'}")
print(f"Saved CA PDB: {ca_pdb}")
print(f"Saved system XML: {output_dir / 'system.xml'}")
print(f"Saved trajectory: {output_dcd}")
print(f"Saved checkpoint: {checkpoint_path}")
print(f"Saved state XML: {output_dir / 'state.xml'}")
print(f"Saved final PDB: {final_pdb}")


Protein: D14
Atomistic PDB: /home/yumzhang/orcd/pool/work/4-idpcg/OpenABC_moff2_test/tutorials/MOFF2-single-protein/D14.pdb
MDP OD CSV: /home/yumzhang/orcd/pool/work/4-idpcg/OpenABC_moff2_test/tutorials/MOFF2-single-protein/MDP_OD_info_0_based.csv
Temperature: 283.15 K
Ionic strength: 156.0 mM
Reference Rg: None nm
Parse molecule with default settings.
Get native pairs with shadow algorithm.
Add protein bonds.
Add protein dihedrals.
Add native pairs.
Use platform: CPU
#"Step","Time (ps)","Potential Energy (kJ/mole)","Kinetic Energy (kJ/mole)","Total Energy (kJ/mole)","Temperature (K)","Speed (ns/day)"
100,1.0000000000000007,-1553.237787867944,1520.8765271506238,-32.36126071732019,253.00055553196128,0
150,1.500000000000001,-1337.4053340813343,1469.3601601666855,131.95482608535121,244.43071489516515,49.9
200,2.0000000000000013,-1253.1044159292765,1598.2503416510942,345.1459257218178,265.8718292368169,49.8
250,2.4999999999999907,-1170.5656669936584,1644.0356136407772,473.46994664711883,27

## OP simulations

In [6]:
# %load /home/yumzhang/orcd/pool/work/4-idpcg/OpenABC_moff2_test/tutorials/MOFF2-single-protein/run_test_OP.py
from pathlib import Path
import json

try:
    import openmm as mm
    import openmm.app as app
    import openmm.unit as unit
except ImportError:
    import simtk.openmm as mm
    import simtk.openmm.app as app
    import simtk.unit as unit

from openabc.forcefields.MOFF2.forcefields import MOFF2Model


# =========================
# Clean user inputs
# =========================

protein = "bba"

reference_dir = Path(
        "/home/yumzhang/orcd/pool/work/4-idpcg/OpenABC_moff2_test/tutorials/MOFF2-single-protein/"
)

aa_pdb = reference_dir / f"{protein}.pdb"

output_dir = Path("outputs") / protein
ca_pdb = output_dir / f"{protein}_ca.pdb"

temperature_K = 300.0
ionic_strength_mM = 150.0
target_rg_nm = None

timestep_fs = 10.0

# Tiny smoke-test values. Increase for production.
n_relax_steps = 50
n_production_steps = 200
report_interval = 50

res_group_mapping_name = "default"
platform_name = "CPU"  # use "CUDA" for GPU production runs


# =========================
# Validate inputs
# =========================

if not aa_pdb.exists():
    raise FileNotFoundError(f"Required input PDB not found: {aa_pdb}")

print(f"Protein: {protein}")
print(f"Atomistic PDB: {aa_pdb}")
print(f"Temperature: {temperature_K} K")
print(f"Ionic strength: {ionic_strength_mM} mM")
print(f"Reference Rg: {target_rg_nm} nm")


# =========================
# Prepare output folder
# =========================

output_dir.mkdir(parents=True, exist_ok=True)

input_parameters = {
    "protein": protein,
    "aa_pdb": str(aa_pdb),
    "ca_pdb": str(ca_pdb),
    "temperature": temperature_K,
    "ionic_strength": ionic_strength_mM,
    "target_rg_nm": target_rg_nm,
    "timestep_fs": timestep_fs,
    "n_relax_steps": n_relax_steps,
    "n_production_steps": n_production_steps,
    "report_interval": report_interval,
    "platform_name": platform_name,
    "res_group_mapping": res_group_mapping_name,
}

with open(output_dir / "input_parameters.json", "w") as f:
    json.dump(input_parameters, f, indent=4)
    f.write("\n")


# =========================
# Build MOFF2 OP system
# =========================

# This constructor:
#   1. parses the atomistic PDB into a CA model;
#   2. computes DSSP;
#   3. keeps native pairs inside continuous ordered H/E segments;
#   4. corrects HIS and terminal charges;
#   5. adds bonds, angles, dihedrals, and native pairs.
model = MOFF2Model.from_folded_pdb(
    aa_pdb=str(aa_pdb),
    ca_pdb=str(ca_pdb),
    bond_force_group=1,
    angle_force_group=2,
    dihedral_force_group=3,
    native_pair_force_group=4,
)

# This method uses packaged parameters:
#   openabc/forcefields/MOFF2/forcefields/parameters/MOFF2.pkl
# and adds:
#   AH + Gaussian contacts, Debye-Huckel electrostatics, density terms.
model.add_moff2_forces(
    temperature=temperature_K,
    ionic_strength=ionic_strength_mM,
    res_group_mapping=res_group_mapping_name,
    contact_force_group=5,
    elec_force_group=6,
    density_force_group_start=7,
)

with open(output_dir / "system.xml", "w") as f:
    f.write(mm.XmlSerializer.serialize(model.system))


# =========================
# Run short simulation
# =========================

T = temperature_K * unit.kelvin
friction_coeff = 1.0 / unit.picosecond
timestep = timestep_fs * unit.femtosecond

integrator = mm.LangevinMiddleIntegrator(T, friction_coeff, timestep)
init_coord = app.PDBFile(str(ca_pdb)).getPositions()

properties = {"Precision": "mixed"} if platform_name == "CUDA" else {}

model.set_simulation(
    integrator,
    platform_name=platform_name,
    init_coord=init_coord,
    properties=properties,
)

model.simulation.minimizeEnergy()
model.simulation.step(n_relax_steps)

output_dcd = output_dir / "output.dcd"
model.add_reporters(report_interval=report_interval, output_dcd=str(output_dcd))

model.simulation.context.setVelocitiesToTemperature(T)
model.simulation.step(n_production_steps)


# =========================
# Save final outputs
# =========================

checkpoint_path = output_dir / "checkpoint.chk"
model.simulation.saveCheckpoint(str(checkpoint_path))

state = model.simulation.context.getState(
    getPositions=True,
    getVelocities=True,
    enforcePeriodicBox=True,
)

with open(output_dir / "state.xml", "w") as f:
    f.write(mm.XmlSerializer.serialize(state))

final_pdb = output_dir / f"{protein}_final.pdb"
with open(final_pdb, "w") as f:
    app.PDBFile.writeFile(
        model.simulation.topology,
        state.getPositions(),
        f,
    )

print(f"Saved input parameters: {output_dir / 'input_parameters.json'}")
print(f"Saved CA PDB: {ca_pdb}")
print(f"Saved system XML: {output_dir / 'system.xml'}")
print(f"Saved trajectory: {output_dcd}")
print(f"Saved checkpoint: {checkpoint_path}")
print(f"Saved state XML: {output_dir / 'state.xml'}")
print(f"Saved final PDB: {final_pdb}")


Protein: bba
Atomistic PDB: /home/yumzhang/orcd/pool/work/4-idpcg/OpenABC_moff2_test/tutorials/MOFF2-single-protein/bba.pdb
Temperature: 300.0 K
Ionic strength: 150.0 mM
Reference Rg: None nm
Parse molecule with default settings.
Get native pairs with shadow algorithm.
Add protein bonds.
Add protein dihedrals.
Add native pairs.
Use platform: CPU
#"Step","Time (ps)","Potential Energy (kJ/mole)","Kinetic Energy (kJ/mole)","Total Energy (kJ/mole)","Temperature (K)","Speed (ns/day)"
100,1.0000000000000007,28.762480678047424,80.25179422119881,109.01427489924623,238.32277252809416,0
150,1.500000000000001,25.251786103834135,103.42954872894121,128.68133483277535,307.1534668305223,43.6
200,2.0000000000000013,82.58207644685959,87.21654873467838,169.79862518153797,259.0059188893446,49.4
250,2.4999999999999907,81.22938346471102,107.1831199754508,188.4125034401618,318.30040148825975,51.9
Saved input parameters: outputs/bba/input_parameters.json
Saved CA PDB: outputs/bba/bba_ca.pdb
Saved system XML:

# MOFF2 LLPS simulations

## NPT preparation

In [8]:
from pathlib import Path
import json
import shutil
import sys

import mdtraj

try:
    import openmm as mm
    import openmm.app as app
    import openmm.unit as unit
except ImportError:
    import simtk.openmm as mm
    import simtk.openmm.app as app
    import simtk.unit as unit

from openabc.forcefields.parsers import HPSParser
from openabc.forcefields.MOFF2.forcefields import MOFF2Model
from openabc.utils.insert import insert_molecules


# =========================
# Clean user inputs
# =========================

protein = "A1-LCD+NLS"

input_pdb = Path(
        f"{protein}_ca.pdb"
)

output_dir = Path(
    f"output/{protein}"
)

temperature_K = 293.0
ionic_strength_mM = 150.0
res_group_mapping_name = "default"

n_mol = 100
box_a_nm = 25.0
box_b_nm = 25.0
box_c_nm = 300.0

timestep_fs = 10.0
n_steps = 5000
output_interval = 500
platform_name = "CPU"  # use "CUDA" for production runs


# =========================
# Validate and prepare files
# =========================

if not input_pdb.exists():
    raise FileNotFoundError(f"Input CA PDB not found: {input_pdb}")

output_dir.mkdir(parents=True, exist_ok=True)

ca_pdb = output_dir / f"{protein}_ca.pdb"
shutil.copyfile(input_pdb, ca_pdb)

input_parameters = {
    "protein": protein,
    "input_pdb": str(input_pdb),
    "temperature": temperature_K,
    "ionic_strength": ionic_strength_mM,
    "res_group_mapping": res_group_mapping_name,
    "n_mol": n_mol,
    "box_a": box_a_nm,
    "box_b": box_b_nm,
    "box_c": box_c_nm,
    "timestep_fs": timestep_fs,
    "n_steps": n_steps,
    "output_interval": output_interval,
    "platform_name": platform_name,
}

with open(output_dir / "input_parameters.json", "w") as f:
    json.dump(input_parameters, f, indent=4)
    f.write("\n")


# =========================
# Build one MOFF2 IDP chain
# =========================

protein_model = MOFF2Model()
protein_model.append_mol(HPSParser(str(ca_pdb)))

protein_model.protein_bonds.loc[:, "k_bond"] = 8000.0
protein_model.protein_bonds.loc[:, "r0"] = 0.386

ca_traj = mdtraj.load_pdb(str(ca_pdb))
assert ca_traj.n_chains == 1

his_mask = protein_model.atoms["resname"] == "HIS"
protein_model.atoms.loc[his_mask, "charge"] = 0.5

charges = protein_model.atoms["charge"].to_numpy()
charges[0] += 1.0
charges[-1] -= 1.0
protein_model.atoms["charge"] = charges


# =========================
# Insert chains into a slab box
# =========================

start_pdb = output_dir / "start.pdb"
if not start_pdb.exists():
    insert_molecules(
        str(ca_pdb),
        str(start_pdb),
        n_mol,
        box=[box_a_nm, box_b_nm, box_c_nm],
    )


# =========================
# Build the MOFF2 LLPS system
# =========================

model = MOFF2Model()
for _ in range(n_mol):
    model.append_mol(protein_model)

top = app.PDBFile(str(start_pdb)).getTopology()
model.create_system(
    top=top,
    box_a=box_a_nm,
    box_b=box_b_nm,
    box_c=box_c_nm,
)

model.add_protein_bonds(force_group=1)
model.add_moff2_forces(
    temperature=temperature_K,
    ionic_strength=ionic_strength_mM,
    res_group_mapping=res_group_mapping_name,
    contact_force_group=2,
    elec_force_group=3,
    density_force_group_start=4,
)

temperature = temperature_K * unit.kelvin
pressure = 1.0 * unit.bar
model.system.addForce(mm.MonteCarloBarostat(pressure, temperature))

with open(output_dir / "system.xml", "w") as f:
    f.write(mm.XmlSerializer.serialize(model.system))


# =========================
# Run a short NPT simulation
# =========================

integrator = mm.LangevinMiddleIntegrator(
    temperature,
    1.0 / unit.picosecond,
    timestep_fs * unit.femtosecond,
)

properties = {"Precision": "mixed"} if platform_name == "CUDA" else {}
init_coord = app.PDBFile(str(start_pdb)).getPositions()

model.set_simulation(
    integrator,
    platform_name=platform_name,
    init_coord=init_coord,
    properties=properties,
)

model.simulation.minimizeEnergy()

output_dcd = output_dir / "output_NPT.dcd"
model.simulation.reporters.append(
    app.DCDReporter(str(output_dcd), output_interval, enforcePeriodicBox=True)
)
model.simulation.reporters.append(
    app.StateDataReporter(
        sys.stdout,
        output_interval,
        step=True,
        potentialEnergy=True,
        kineticEnergy=True,
        totalEnergy=True,
        temperature=True,
        speed=True,
    )
)

model.simulation.context.setVelocitiesToTemperature(temperature)

for start in range(0, n_steps, output_interval):
    model.simulation.step(output_interval)
    state = model.simulation.context.getState(
        getPositions=False,
        enforcePeriodicBox=True,
    )
    box = state.getPeriodicBoxVectors(asNumpy=True).value_in_unit(unit.nanometer)
    print(
        f"Step {start + output_interval}: "
        f"box=({box[0][0]:.2f}, {box[1][1]:.2f}, {box[2][2]:.2f}) nm"
    )

final_state = model.simulation.context.getState(
    getPositions=True,
    getVelocities=True,
    enforcePeriodicBox=True,
)

with open(output_dir / "NPT_final_state.xml", "w") as f:
    f.write(mm.XmlSerializer.serialize(final_state))

final_pdb = output_dir / f"{protein}_final.pdb"
with open(final_pdb, "w") as f:
    app.PDBFile.writeFile(
        model.simulation.topology,
        final_state.getPositions(),
        f,
    )

print(f"Saved input parameters: {output_dir / 'input_parameters.json'}")
print(f"Saved start structure: {start_pdb}")
print(f"Saved system XML: {output_dir / 'system.xml'}")
print(f"Saved trajectory: {output_dcd}")
print(f"Saved final state XML: {output_dir / 'NPT_final_state.xml'}")
print(f"Saved final PDB: {final_pdb}")


In [9]:
# %load /home/yumzhang/orcd/pool/work/4-idpcg/OpenABC_moff2_test/tutorials/MOFF2-IDP-LLPS/run_slab.py
#!/usr/bin/env python3
import os, sys, argparse, numpy as np
try:
    import openmm as mm
    import openmm.app as app
    import openmm.unit as unit
except ImportError:
    import simtk.openmm as mm
    import simtk.openmm.app as app
    import simtk.unit as unit
import pickle
import mdtraj
import shutil
import glob
import json
import sys

protein='A1-LCD+NLS'
output_dir=f'output/{protein}'


system_path = os.path.join(f"{output_dir}/system.xml")
state_path = os.path.join(f"{output_dir}/NPT_final_state.xml")
top_path = os.path.join(f"{output_dir}/start.pdb")
temperature = 298.0
timestep = 10.0
n_steps = 5000
output_interval = 500
box_a=25
box_b=25
box_c=300

with open(system_path) as f:
    system = mm.XmlSerializer.deserialize(f.read())
with open(state_path) as f:
    npt_state = mm.XmlSerializer.deserialize(f.read())

# Remove barostat
for i, fce in enumerate(list(system.getForces())):
    if isinstance(fce, mm.MonteCarloBarostat):
        system.removeForce(i)
        print(f"Removed barostat (force {i})")

top = app.PDBFile(top_path).getTopology()
integrator = mm.NoseHooverIntegrator(temperature * unit.kelvin,
                                     0.01 / unit.picosecond,
                                     timestep * unit.femtosecond)
platform = mm.Platform.getPlatformByName("CPU")
properties = {"Precision": "mixed"} if platform == "CUDA" else {}
sim = app.Simulation(top, system, integrator, platform, properties)

# Load previous state
sim.context.setPeriodicBoxVectors(*npt_state.getPeriodicBoxVectors())
sim.context.setPositions(npt_state.getPositions())
vel = npt_state.getVelocities()
if vel is not None:
    sim.context.setVelocities(vel)
else:
    sim.context.setVelocitiesToTemperature(temperature * unit.kelvin)

# Override box if requested
if (box_a, box_b, box_c) != (0, 0, 0):
    new_a = mm.Vec3(box_a, 0, 0) * unit.nanometer
    new_b = mm.Vec3(0, box_b, 0) * unit.nanometer
    new_c = mm.Vec3(0, 0, box_c) * unit.nanometer
    sim.context.setPeriodicBoxVectors(new_a, new_b, new_c)
    print(f"Set new box: {box_a} × {box_b} × {box_c} nm")

    # Recenter positions (keep units)
    pos_np = sim.context.getState(getPositions=True).getPositions(asNumpy=True)
    center = 0.5 * np.array([box_a, box_b, box_c]) * unit.nanometer
    pos_np -= np.mean(pos_np, axis=0)
    pos_np += center
    sim.context.setPositions(pos_np)
    print("Recentered coordinates into new box.")

# Minimize and run NVT
sim.minimizeEnergy(maxIterations=200)
# Reporters
output_dcd = os.path.join(output_dir, f"slab.dcd")
dcd_reporter = app.DCDReporter(output_dcd, output_interval, enforcePeriodicBox=True)
state_data_reporter = app.StateDataReporter(
    sys.stdout, output_interval,
    step=True, time=True, potentialEnergy=True,
    kineticEnergy=True, totalEnergy=True,
    temperature=True, speed=True
)
sim.reporters.append(dcd_reporter)
sim.reporters.append(state_data_reporter)

checkpoint_interval = 50000
chk_path = os.path.join(output_dir, f"chk.chk")
xml_prefix = os.path.join(output_dir, f"chk")

for step in range(0, n_steps, checkpoint_interval):
    sim.step(checkpoint_interval)
    sim.saveCheckpoint(chk_path)
    state = sim.context.getState(
        getPositions=True, 
        getVelocities=True, 
        enforcePeriodicBox=True
    )
    xml_path = f"{xml_prefix}.xml"
    with open(xml_path, "w") as f:
        f.write(mm.XmlSerializer.serialize(state))

print("✅ Simulation completed successfully.")




## Production

In [10]:
# %load /home/yumzhang/orcd/pool/work/4-idpcg/OpenABC_moff2_test/tutorials/MOFF2-IDP-LLPS/run_slab.py
#!/usr/bin/env python3
import os, sys, argparse, numpy as np
try:
    import openmm as mm
    import openmm.app as app
    import openmm.unit as unit
except ImportError:
    import simtk.openmm as mm
    import simtk.openmm.app as app
    import simtk.unit as unit
import pickle
import mdtraj
import shutil
import glob
import json
import sys

protein='A1-LCD+NLS'
output_dir=f'output/{protein}'


system_path = os.path.join(f"{output_dir}/system.xml")
state_path = os.path.join(f"{output_dir}/NPT_final_state.xml")
top_path = os.path.join(f"{output_dir}/start.pdb")
temperature = 298.0
timestep = 10.0
n_steps = 5000
output_interval = 500
box_a=25
box_b=25
box_c=300

with open(system_path) as f:
    system = mm.XmlSerializer.deserialize(f.read())
with open(state_path) as f:
    npt_state = mm.XmlSerializer.deserialize(f.read())

# Remove barostat
for i, fce in enumerate(list(system.getForces())):
    if isinstance(fce, mm.MonteCarloBarostat):
        system.removeForce(i)
        print(f"Removed barostat (force {i})")

top = app.PDBFile(top_path).getTopology()
integrator = mm.NoseHooverIntegrator(temperature * unit.kelvin,
                                     0.01 / unit.picosecond,
                                     timestep * unit.femtosecond)
platform = mm.Platform.getPlatformByName("CPU")
properties = {"Precision": "mixed"} if platform == "CUDA" else {}
sim = app.Simulation(top, system, integrator, platform, properties)

# Load previous state
sim.context.setPeriodicBoxVectors(*npt_state.getPeriodicBoxVectors())
sim.context.setPositions(npt_state.getPositions())
vel = npt_state.getVelocities()
if vel is not None:
    sim.context.setVelocities(vel)
else:
    sim.context.setVelocitiesToTemperature(temperature * unit.kelvin)

# Override box if requested
if (box_a, box_b, box_c) != (0, 0, 0):
    new_a = mm.Vec3(box_a, 0, 0) * unit.nanometer
    new_b = mm.Vec3(0, box_b, 0) * unit.nanometer
    new_c = mm.Vec3(0, 0, box_c) * unit.nanometer
    sim.context.setPeriodicBoxVectors(new_a, new_b, new_c)
    print(f"Set new box: {box_a} × {box_b} × {box_c} nm")

    # Recenter positions (keep units)
    pos_np = sim.context.getState(getPositions=True).getPositions(asNumpy=True)
    center = 0.5 * np.array([box_a, box_b, box_c]) * unit.nanometer
    pos_np -= np.mean(pos_np, axis=0)
    pos_np += center
    sim.context.setPositions(pos_np)
    print("Recentered coordinates into new box.")

# Minimize and run NVT
sim.minimizeEnergy(maxIterations=200)
# Reporters
output_dcd = os.path.join(output_dir, f"slab.dcd")
dcd_reporter = app.DCDReporter(output_dcd, output_interval, enforcePeriodicBox=True)
state_data_reporter = app.StateDataReporter(
    sys.stdout, output_interval,
    step=True, time=True, potentialEnergy=True,
    kineticEnergy=True, totalEnergy=True,
    temperature=True, speed=True
)
sim.reporters.append(dcd_reporter)
sim.reporters.append(state_data_reporter)

checkpoint_interval = 50000
chk_path = os.path.join(output_dir, f"chk.chk")
xml_prefix = os.path.join(output_dir, f"chk")

for step in range(0, n_steps, checkpoint_interval):
    sim.step(checkpoint_interval)
    sim.saveCheckpoint(chk_path)
    state = sim.context.getState(
        getPositions=True, 
        getVelocities=True, 
        enforcePeriodicBox=True
    )
    xml_path = f"{xml_prefix}.xml"
    with open(xml_path, "w") as f:
        f.write(mm.XmlSerializer.serialize(state))

print("✅ Simulation completed successfully.")


